# notebook 19 — 連続回転 SO(3) 対称性の起源：T³ の連続極限

nb18 (c) は、離散 T³ の等長群が並進と軸置換 S₃ のみで、**連続 SO(3) 回転は等長でない**ことを
確定した（D-10）。だが物理空間は連続回転対称を持つ。この notebook はその開いた問い——
**T³ の連続極限 ε→0（本丸の連続極限と直結）で連続回転 SO(3) 対称が創発するか**——に答える。

**核心の警戒（D-10）**：「連続極限を取れば都合よく対称性が出る」という期待は後付けの罠になりうる。
本 notebook は破れの大きさを ε→0 で追うだけでなく、**球対称核との対照実験**と**小角度展開の解析**で、
破れがどの次数に属するか（計量レベルか高次補正か）を構造的に分離する。創発するなら*どのレベルで*
創発し*何が残る*かまで誠実に画定する。

**先取りした結論**：連続回転対称は**計量レベル（2次）では創発する**が、**高次（4次以上）に格子の
立方異方性が残る**。これは格子場理論で連続極限が回転対称を回復する標準的描像と整合する。

## 19.0 セットアップ

In [1]:
import numpy as np
import sympy as sp
np.set_printoptions(precision=5, suppress=True, linewidth=120)

N = 3  # 与件（QCD との同定）

def f_axis(x):
    # 単一軸の entanglement 距離 d = -log(|cos2x|/(2N))。発散点 x=π/4 を避けて使う。
    a = np.abs(np.cos(2*x))/(2*N)
    return -np.log(a)

def kern_sep(v):
    # 我々の距離核（各軸独立、加法分解）= Σ_c d(Δφ_c)^2
    return sum(f_axis(x)**2 for x in v)

def kern_sph(v):
    # 対照用：球対称核 g(|v|)（回転不変な仮想核）。破れの基準線。
    r = np.linalg.norm(v)
    a = np.abs(np.cos(2*r))/(2*N)
    return (-np.log(a))**2

def rot(axis, ang):
    c, s = np.cos(ang), np.sin(ang)
    if axis == 0: return np.array([[1,0,0],[0,c,-s],[0,s,c]])
    if axis == 1: return np.array([[c,0,s],[0,1,0],[-s,0,c]])
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

print("setup done. N =", N)

setup done. N = 3


## 19.1 ナイーブな指標の罠：発散点が連続極限を汚染する

まず素朴に「格子点数 n_per を増やして連続 SO(3) 回転の距離破れを測る」とどうなるかを見る。
これは破れが 0 に収束する*はず*という期待で行うが、`cos2θ=0`（x=π/4）の発散点を格子が細かいほど
拾い、指標が暴れる。**この素朴な指標は信頼できない**ことを明示し、次節で作り直す動機とする。

In [2]:
def dmat(points):
    d2 = np.zeros((len(points), len(points)))
    for c in range(3):
        dphi = points[:,c][:,None]-points[:,c][None,:]
        a = np.abs(np.cos(2*dphi))/(2*N)
        with np.errstate(divide="ignore"):
            d = -np.log(a)
        d[~np.isfinite(d)] = 0.0; np.fill_diagonal(d, 0.0)
        d2 += d**2
    return np.sqrt(d2)

def grid(n_per):
    ph = np.linspace(0, 2*np.pi, n_per, endpoint=False)
    P = np.meshgrid(ph, ph, ph, indexing="ij")
    return np.stack([p.ravel() for p in P], 1)

print(f"  {'n_per':>6} {'ε~2π/n':>10} {'rel破れ':>12}")
for n_per in [4, 6, 8, 10, 12]:
    sp_grid = grid(n_per); D0 = dmat(sp_grid)
    Dr = dmat(sp_grid @ rot(0, 0.5).T)
    rel = np.abs(Dr-D0).max()/D0[D0>0].mean()
    print(f"  {n_per:6d} {2*np.pi/n_per:10.4f} {rel:12.4f}")
print("  → 収束せず暴れる。発散点 x=π/4 を格子が拾うアーティファクト。指標を作り直す。")

   n_per     ε~2π/n        rel破れ
       4     1.5708       0.6815
       6     1.0472       0.4744
       8     0.7854       1.1883


      10     0.6283       1.0403


      12     0.5236       1.8938
  → 収束せず暴れる。発散点 x=π/4 を格子が拾うアーティファクト。指標を作り直す。


## 19.2 対照実験：分離核 vs 球対称核（破れの起源を格子から分離）

発散点から遠い小角度 Δφ に限定し、回転による核の相対変化を測る。同時に**球対称核**（定義上
回転不変）を対照に置く。球対称核が常に不変（基準線が正しい）なことを確認し、分離核の破れが
「格子の粗さ」でなく「核が各軸独立＝非球対称」であることに由来すると示す。

In [3]:
rng = np.random.default_rng(1)
print("  |Δφ|      分離核 相対変化     球対称核(対照)")
for amp in [0.5, 0.2, 0.1, 0.05, 0.02]:
    sep, sph = [], []
    for _ in range(300):
        v = rng.normal(size=3); v = v/np.linalg.norm(v)*amp
        R = rot(rng.integers(3), rng.uniform(0, np.pi))
        sep.append(abs(kern_sep(R@v)-kern_sep(v))/max(kern_sep(v), 1e-12))
        sph.append(abs(kern_sph(R@v)-kern_sph(v))/max(kern_sph(v), 1e-12))
    print(f"  {amp:5.2f}     {np.mean(sep):.5f}          {np.mean(sph):.2e}")
print()
print("  → 球対称核は常に不変（~0、対照が正しい）。分離核は有限の破れ。")
print("    破れの起源は格子でなく『核が球対称でない（各軸独立周期）』こと。")

  |Δφ|      分離核 相対変化     球対称核(対照)
   0.50     0.00786          2.15e-17
   0.20     0.00017          0.00e+00
   0.10     0.00001          0.00e+00
   0.05     0.00000          1.15e-17
   0.02     0.00000          0.00e+00

  → 球対称核は常に不変（~0、対照が正しい）。分離核は有限の破れ。
    破れの起源は格子でなく『核が球対称でない（各軸独立周期）』こと。


## 19.3 破れの小角度スケーリング：どの次数に属するか

破れ（分離核の回転による変化）を |Δφ| の冪で規格化し、何次の項かを特定する。/|Δφ|⁴ が一定に
収束すれば、破れは O(|Δφ|⁴) で**最低次（計量、2次）は回転不変**だと分かる。

In [4]:
print("  |Δφ|       <|Δkern|>      /|Δφ|²      /|Δφ|⁴")
for amp in [0.2, 0.1, 0.05, 0.025, 0.0125]:
    diffs = []
    for _ in range(2000):
        v = rng.normal(size=3); v = v/np.linalg.norm(v)*amp
        R = rot(rng.integers(3), rng.uniform(0, np.pi))
        diffs.append(kern_sep(R@v)-kern_sep(v))
    m = np.mean(np.abs(diffs))
    print(f"  {amp:7.4f}   {m:.3e}    {m/amp**2:.4f}    {m/amp**4:.3f}")
print()
print("  → /|Δφ|⁴ が ~1.0 に収束：破れは O(|Δφ|⁴) の高次項。")
print("    /|Δφ|² は 0 へ → 最低次（計量）に破れなし＝計量は連続回転不変。")

  |Δφ|       <|Δkern|>      /|Δφ|²      /|Δφ|⁴
   0.2000   1.628e-03    0.0407    1.017
   0.1000   9.965e-05    0.0100    0.997


   0.0500   5.952e-06    0.0024    0.952
   0.0250   3.835e-07    0.0006    0.982


   0.0125   2.251e-08    0.0001    0.922

  → /|Δφ|⁴ が ~1.0 に収束：破れは O(|Δφ|⁴) の高次項。
    /|Δφ|² は 0 へ → 最低次（計量）に破れなし＝計量は連続回転不変。


## 19.4 解析的確認：距離核の小角度展開

数値の結論を解析で裏づける。単一軸核 `−log(|cos2x|/(2N))` を x→0 で級数展開し、各次数が
回転対称かを判定する。

In [5]:
x = sp.symbols("x", real=True)
ser = sp.series(-sp.log(sp.cos(2*x)/(2*N)), x, 0, 7).removeO()
print("単一軸核 −log(|cos2x|/(2N)) の小角度展開：")
print(" ", ser)
print()
print("和 Σ_c [ log6 + 2Δφ_c² + (4/3)Δφ_c⁴ + (64/45)Δφ_c⁶ + ... ]：")
print("  ・定数項 Σ log6        : 回転不変（自明）")
print("  ・2次 2Σφ_c² = 2|Δφ|²  : 球対称 → 連続 SO(3) 不変（計量は等方的）")
print("  ・4次 (4/3)Σφ_c⁴       : Σφ_c⁴ ≠ |Δφ|⁴/系数 → 立方異方性（回転対称を破る）")
print()
print("→ 計量（2次）は SO(3) 不変。破れは4次以降の立方異方性（格子の方向依存）。")

単一軸核 −log(|cos2x|/(2N)) の小角度展開：
  64*x**6/45 + 4*x**4/3 + 2*x**2 + log(6)

和 Σ_c [ log6 + 2Δφ_c² + (4/3)Δφ_c⁴ + (64/45)Δφ_c⁶ + ... ]：
  ・定数項 Σ log6        : 回転不変（自明）
  ・2次 2Σφ_c² = 2|Δφ|²  : 球対称 → 連続 SO(3) 不変（計量は等方的）
  ・4次 (4/3)Σφ_c⁴       : Σφ_c⁴ ≠ |Δφ|⁴/系数 → 立方異方性（回転対称を破る）

→ 計量（2次）は SO(3) 不変。破れは4次以降の立方異方性（格子の方向依存）。


## 19.5 nb18 の否定との整合：見ていたレベルが違う

nb18 (c) は「連続 SO(3) は等長でない」と否定した。本 notebook は「計量レベルでは回転不変」と
肯定する。矛盾ではなく、**見ていたレベルが違う**。nb18 は核全体（全次数）の等長性を見たので、
4次の立方異方性を拾って「等長でない」と正しく判定した。本 notebook はそれを次数分解し、最低次
（計量）は不変・高次が異方、と精密化した。両者は整合する。

In [6]:
print("nb18 (c)：核全体（全次数）の等長性 → 4次異方性を拾い『連続 SO(3) 非等長』（正しい）")
print("nb19    ：次数分解 → 2次（計量）は SO(3) 不変、4次以降が立方異方（精密化）")
print()
print("整合：両者は同じ核を異なる解像度で見た。nb18 は『全体は非対称』、")
print("      nb19 は『最低次は対称・高次が破る』。後者が前者を分解して包含する。")

nb18 (c)：核全体（全次数）の等長性 → 4次異方性を拾い『連続 SO(3) 非等長』（正しい）
nb19    ：次数分解 → 2次（計量）は SO(3) 不変、4次以降が立方異方（精密化）

整合：両者は同じ核を異なる解像度で見た。nb18 は『全体は非対称』、
      nb19 は『最低次は対称・高次が破る』。後者が前者を分解して包含する。


## 19.6 サマリー（established / open）

### established（この notebook で確定）

1. **ナイーブな連続極限指標は発散点に汚染される（19.1）。** 格子点数を増やす素朴な破れ指標は
   `cos2θ=0` を拾って暴れ、信頼できない。対照実験と小角度解析で作り直す必要がある。

2. **破れの起源は核の非球対称性であって格子の粗さではない（19.2）。** 球対称核（対照）は厳密に
   回転不変。分離核（各軸独立周期）のみ有限の破れを示す。破れは離散化アーティファクトではなく
   距離核の構造に内在する。

3. **連続回転対称は計量レベルで創発する（19.3–19.4、条件つき肯定）。** 距離核の小角度展開は
   `2|Δφ|² + (4/3)Σφ_c⁴ + …`。最低次（2次＝計量）は球対称 `2|Δφ|²` で**連続 SO(3) 不変**。
   破れは O(|Δφ|⁴) の立方異方性（4次以降）。数値（/|Δφ|⁴→1.0）と解析が一致。

4. **高次に格子の立方異方性が残る（19.4）。** 4次項 `Σφ_c⁴` は方向依存（立方対称のみ）で連続回転を
   破る。これは格子場理論で連続極限が低エネルギー（長波長＝小角度）で回転対称を回復し、高次
   （格子スケール）に異方性を残す標準的描像と整合する。

5. **nb18 の否定との整合（19.5）。** nb18 は核全体を見て「連続 SO(3) 非等長」と正しく判定。nb19 は
   次数分解して「計量は不変・高次が異方」と精密化。後者が前者を包含する。

### 連続回転対称の到達点

| レベル | 連続 SO(3) 対称 | 根拠 |
|---|---|---|
| 計量（2次, 長波長） | **創発する（不変）** | 核の2次項 = 2\|Δφ\|² が球対称（19.3–19.4） |
| 高次（4次以上, 格子スケール） | 破れる（立方異方） | Σφ_c⁴ の方向依存（19.4） |
| 離散・全次数（nb18） | 非等長 | 全次数で見れば4次異方を拾う（19.5） |

**到達点**：連続回転 SO(3) 対称は「ε→0 で都合よく丸ごと出る」のではなく、**長波長（計量）レベルで
創発し、格子スケールに立方異方性を残す**という、量子場理論的に自然な形で創発する。これは
nb18 (c) の否定を覆すのではなく、その解像度を上げて「どのレベルで何が成り立つか」を画定したもの。

### open（次へ）

1. **立方異方性は物理的に許容範囲か（新規）。** 残る4次異方性が現実の回転対称の精密検証
   （等方性の実験限界）と整合するか。格子間隔 ε（本丸の無限小格子）が十分小さければ高次異方は
   抑制される——ε のオーダーと異方性の大きさの定量関係が次の課題。

2. **「実空間3次元」の上流原理（nb17/README §5 第二段階、継続）。** 計量レベルで SO(3) 対称が
   出ても、軸数3（空間次元3）自体は依然 nb17 の外部入力。次元の起源は未解決のまま。

3. **本丸 no-go の解析的裏づけ（C-1、継続）。**

### 教訓（D への追記候補）

- **D-19（候補）：対称性の創発は『レベル（次数・スケール）』を指定して論じよ。** 「連続極限で対称性が
  創発するか」は yes/no では誤る。計量（長波長）では創発し高次（格子スケール）では破れる、という
  ように*どのレベルで*成り立つかを次数分解して画定する。nb18 の全次数否定と nb19 の計量レベル肯定は
  解像度の違いで両立する。

- **D-20（候補）：連続極限の指標は発散点・特異点に汚染されうる。** 素朴な「格子を細かくして破れを測る」
  指標は核の特異点（cos2θ=0）を拾って暴れる。対照核（既知の対称性を持つ基準）と小角度展開で、
  破れの真の起源（構造か離散化か）を分離してから結論する。